In [1]:
import time
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [ ]:
# ---------- 1) Load + chunk ----------
def load_chunks(path: str, mode: str = "lines", min_chars: int = 20) -> list[str]:
    """
    mode:
      - lines: each non-empty line is a chunk (fastest, simplest)
      - paragraphs: split by blank lines (better for prose)
    """
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    if mode == "lines":
        chunks = [ln.strip() for ln in text.splitlines() if ln.strip()]
    elif mode == "paragraphs":
        chunks = [p.strip() for p in text.split("\n\n") if p.strip()]
    else:
        raise ValueError("mode must be 'lines' or 'paragraphs'")

    # filter low-information chunks (prevents junk vectors)
    chunks = [c for c in chunks if len(c) >= min_chars]
    return chunks

In [3]:
docs = load_chunks("sample.txt", mode="lines", min_chars=20)
print("Chunks:", len(docs))
print("Example chunk:", docs[0] if docs else "EMPTY")

Chunks: 1716
Example chunk: ArticleTitle	Question	Answer	DifficultyFromQuestioner	DifficultyFromAnswerer	ArticleFile


In [4]:
# ---------- 2) Load embedding model ----------
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# ---------- 3) Embed corpus ----------
t0 = time.time()
doc_emb = model.encode(docs, normalize_embeddings=True)  # shape: (N, D)
doc_emb = np.asarray(doc_emb, dtype="float32")
print(f"Embedded {len(docs)} chunks in {(time.time()-t0):.2f}s | shape={doc_emb.shape}")

Embedded 1716 chunks in 15.20s | shape=(1716, 384)


In [6]:
# ---------- 4) Build FAISS Flat IP index (cosine via normalization) ----------
D = doc_emb.shape[1]
index = faiss.IndexFlatIP(D)
index.add(doc_emb)
assert index.ntotal == len(docs), "Index size mismatch"
print("FAISS index vectors:", index.ntotal)

FAISS index vectors: 1716


In [7]:
# ---------- 5) NumPy brute-force search (ground truth) ----------
def search_numpy(query: str, top_k: int = 5):
    q = model.encode([query], normalize_embeddings=True)[0].astype("float32")  # shape: (D,)
    t0 = time.time()
    scores = doc_emb @ q                         # (N,)
    idx = np.argsort(-scores)[:top_k]            # top-k
    latency_ms = (time.time() - t0) * 1000
    return latency_ms, [(int(i), float(scores[i]), docs[int(i)]) for i in idx]

In [8]:
# ---------- 6) FAISS search ----------
def search_faiss(query: str, top_k: int = 5):
    q = model.encode([query], normalize_embeddings=True).astype("float32")  # shape: (1, D)
    t0 = time.time()
    scores, idx = index.search(q, top_k)  # each: (1, top_k)
    latency_ms = (time.time() - t0) * 1000
    results = [(int(i), float(s), docs[int(i)]) for i, s in zip(idx[0], scores[0])]
    return latency_ms, results

In [9]:
# ---------- 7) Test + compare ----------
query = "How do turtles reproduce?"
k = 5

lat_np, res_np = search_numpy(query, top_k=k)
lat_fa, res_fa = search_faiss(query, top_k=k)

print("\n--- NumPy brute-force ---")
print(f"Latency: {lat_np:.3f} ms")
for i, s, d in res_np:
    print(f"[{i}] score={s:.4f} | {d}")

print("\n--- FAISS FlatIP ---")
print(f"Latency: {lat_fa:.3f} ms")
for i, s, d in res_fa:
    print(f"[{i}] score={s:.4f} | {d}")


--- NumPy brute-force ---
Latency: 3.711 ms
[1522] score=0.8214 | turtle	How do turtles reproduce?	They lay eggs	hard	too hard	S08_set1_a9
[1523] score=0.8173 | turtle	How do turtles reproduce?	they lay eggs	hard	hard	S08_set1_a9
[1544] score=0.6574 | turtle	Do turtles lay eggs underwater?	No	easy	easy	S08_set1_a9
[1545] score=0.6574 | turtle	Do turtles lay eggs underwater?	no	easy	easy	S08_set1_a9
[1569] score=0.6478 | turtle	Can turtles take many years to reach breeding age ?	yes	NULL	easy	S08_set1_a9

--- FAISS FlatIP ---
Latency: 2.266 ms
[1522] score=0.8214 | turtle	How do turtles reproduce?	They lay eggs	hard	too hard	S08_set1_a9
[1523] score=0.8173 | turtle	How do turtles reproduce?	they lay eggs	hard	hard	S08_set1_a9
[1545] score=0.6574 | turtle	Do turtles lay eggs underwater?	no	easy	easy	S08_set1_a9
[1544] score=0.6574 | turtle	Do turtles lay eggs underwater?	No	easy	easy	S08_set1_a9
[1569] score=0.6478 | turtle	Can turtles take many years to reach breeding age ?	yes	NULL	ea

In [10]:
# Optional: check if top-k indices match
np_idx = [x[0] for x in res_np]
fa_idx = [x[0] for x in res_fa]
print("\nTop-k index match:", np_idx == fa_idx)


Top-k index match: False


## HNSW implementation

In [11]:
# doc_emb: (N, D), float32, normalized
D = doc_emb.shape[1]

# --- HNSW parameters ---
M = 32                  # max neighbors per node
efConstruction = 200    # build quality

index_hnsw = faiss.IndexHNSWFlat(D, M, faiss.METRIC_INNER_PRODUCT)
index_hnsw.hnsw.efConstruction = efConstruction

# add vectors
index_hnsw.add(doc_emb)

print("HNSW index size:", index_hnsw.ntotal)

HNSW index size: 1716


In [12]:
def search_hnsw(query: str, top_k: int = 5, efSearch: int = 64):
    index_hnsw.hnsw.efSearch = efSearch  # query-time knob

    q = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index_hnsw.search(q, top_k)

    return [(int(i), float(s), docs[int(i)]) for i, s in zip(idx[0], scores[0])]


In [13]:
def recall_at_k(flat_idx, ann_idx, k):
    return len(set(flat_idx[:k]) & set(ann_idx[:k])) / k

In [14]:
query = "How do turtles reproduce?"
k = 100

# FAISS Flat (ground truth)
_, flat_res = search_faiss(query, top_k=k)
flat_idx = [i for i, _, _ in flat_res]

for ef in [8, 16, 32, 64, 128]:
    ann_res = search_hnsw(query, top_k=k, efSearch=ef)
    ann_idx = [i for i, _, _ in ann_res]

    r = recall_at_k(flat_idx, ann_idx, k)
    print(f"efSearch={ef:>3} | Recall@{k}={r:.2f}")

efSearch=  8 | Recall@100=0.54
efSearch= 16 | Recall@100=0.68
efSearch= 32 | Recall@100=0.85
efSearch= 64 | Recall@100=0.91
efSearch=128 | Recall@100=1.00


## IVF indexing

In [15]:
N, D = doc_emb.shape

# number of clusters
nlist = int(np.sqrt(N))
nlist = max(64, min(nlist, 4096))

# quantizer
quantizer = faiss.IndexFlatIP(D)

# IVF index
index_ivf = faiss.IndexIVFFlat(
    quantizer,
    D,
    nlist,
    faiss.METRIC_INNER_PRODUCT
)

# TRAINING STEP (required)
index_ivf.train(doc_emb)

# add vectors
index_ivf.add(doc_emb)

# search parameter
index_ivf.nprobe = 10


In [16]:
def search_ivf(query, k=5):

    q = model.encode([query], normalize_embeddings=True).astype("float32")

    scores, idx = index_ivf.search(q, k)

    results = [(int(i), float(scores[0][j]), docs[int(i)])
               for j, i in enumerate(idx[0])]

    return results


In [20]:
query = "How do turtles reproduce?"

_, flat_results = search_faiss(query, 5)
ivf_results = search_ivf(query, 5)

flat_idx = [i for i, _, _ in flat_results]
ivf_idx = [i for i, _, _ in ivf_results]

recall = len(set(flat_idx) & set(ivf_idx)) / 5

print("Recall@5:", recall)

Recall@5: 1.0
